# Phase 17 — Validation-strategy comparison (ordinary vs group-aware)
**Resolves:** R2-10 (the 0.56 -> 0.84 Pakistan result "should be documented more comprehensively, ideally with a
dedicated table showing performance for all relevant validation strategies"); R2-9(b) (reproduction evidence).

For every Study 2 dataset, using the pooled analytic sample and TWO random-forest configurations:
 * `tuned`  = the hyperparameters selected under group-aware CV (notebooks 05a-c; typically shallow trees)
 * `default`= 300 trees, unlimited depth (a high-capacity forest that can memorise duplicate profiles)
four validation strategies are compared:
 S1 ordinary stratified 5-fold CV · S2 group-aware 5-fold CV · S3 ordinary random 60/20/20 split (test AUC) ·
 S4 the study's group-aware held-out test partition.
"Exposure" = share of validation rows whose exact feature signature also occurs in the training rows.
No hyperparameters are retuned here, so differences are due to the validation design only.
**Outputs:** `S2_validation_strategy_table.csv`

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from revision_utils import *

ENV = get_env()          # paths + config (notebooks must be run from the notebooks/ folder)
SEED = ENV["SEED"]       # 42
B = ENV["N_BOOT"]        # 2000 bootstrap resamples (config.yaml)
print("Project root:", ENV["ROOT"], "| outputs ->", ENV["REV"])

Project root: C:\maternal-mental-health-risk | outputs -> C:\maternal-mental-health-risk\reports/tables\revision


In [2]:
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from modeling import build_model_pipeline
from preprocessing import compute_group_id
B17 = min(B, 1000)

def run_cv(factory, X, y, sig, splitter, use_groups):
    oof = np.zeros(len(y)); expo, fold_auc = [], []
    it = splitter.split(X, y, sig) if use_groups else splitter.split(X, y)
    for tr, va in it:
        pipe = factory(); pipe.fit(X.iloc[tr], y.iloc[tr])
        oof[va] = pipe.predict_proba(X.iloc[va])[:, 1]
        trs = set(sig.iloc[tr]); expo.append(float(np.mean([s in trs for s in sig.iloc[va]])))
        fold_auc.append(safe_auc(y.iloc[va], oof[va]))
    return oof, float(np.mean(expo)), fold_auc

rows = []
for tag, d in DATASETS.items():
    S = load_split(tag, ENV); feats, lab = features_of(tag), d["label"]
    full = pd.concat([S["train"], S["cal"], S["test"]], ignore_index=True)
    X, y = full[feats], full[lab].astype(int)
    sig = compute_group_id(X, feats)
    tuned = load_rf_params(tag, ENV)["rf"]
    configs = {"tuned": tuned, "default": {"n_estimators": 300}}
    for cname, params in configs.items():
        fac = lambda: build_model_pipeline("rf", params, d["numeric"], d["categorical"], SEED)
        # S1 ordinary CV
        oof1, e1, fa1 = run_cv(fac, X, y, sig, StratifiedKFold(5, shuffle=True, random_state=SEED), False)
        # S2 group-aware CV
        oof2, e2, fa2 = run_cv(fac, X, y, sig, StratifiedGroupKFold(5, shuffle=True, random_state=SEED), True)
        for sname, oof, e, fa in [("S1 ordinary stratified 5-fold CV", oof1, e1, fa1),
                                  ("S2 group-aware 5-fold CV", oof2, e2, fa2)]:
            est, lo, hi = bootstrap_ci_full(y.values, oof, B=B17, seed=SEED)
            rows.append(dict(dataset=tag, rf_config=cname, strategy=sname, n_eval=len(y), auc=est, ci_lower=lo, ci_upper=hi,
                             fold_mean_auc=np.mean(fa), fold_sd_auc=np.std(fa, ddof=1), signature_exposure=e))
        # S3 ordinary random split
        idx = np.arange(len(full))
        tr_i, te_i = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)
        pipe = fac(); pipe.fit(X.iloc[tr_i], y.iloc[tr_i]); p3 = pipe.predict_proba(X.iloc[te_i])[:, 1]
        est, lo, hi = bootstrap_ci_full(y.iloc[te_i].values, p3, B=B17, seed=SEED)
        trs = set(sig.iloc[tr_i])
        rows.append(dict(dataset=tag, rf_config=cname, strategy="S3 ordinary random 80/20 split (test AUC)", n_eval=len(te_i),
                         auc=est, ci_lower=lo, ci_upper=hi, signature_exposure=float(np.mean([s in trs for s in sig.iloc[te_i]]))))
        # S4 group-aware held-out partition (the study design)
        Xtr, ytr = S["train"][feats], S["train"][lab].astype(int)
        Xte, yte = S["test"][feats], S["test"][lab].astype(int)
        pipe = fac(); pipe.fit(Xtr, ytr); p4 = pipe.predict_proba(Xte)[:, 1]
        est, lo, hi = bootstrap_ci_full(yte.values, p4, B=B17, seed=SEED)
        trs = set(compute_group_id(Xtr, feats)); sg = compute_group_id(Xte, feats)
        rows.append(dict(dataset=tag, rf_config=cname, strategy="S4 group-aware held-out test partition (study design)", n_eval=len(yte),
                         auc=est, ci_lower=lo, ci_upper=hi, signature_exposure=float(np.mean([s in trs for s in sg]))))
        print(tag, cname, "done")
val = pd.DataFrame(rows)
save_table(val, ENV, "S2_validation_strategy_table.csv")
val.round(3)

pakistan tuned done
pakistan default done
arab5_depression tuned done
arab5_depression default done
arab5_anxiety tuned done
arab5_anxiety default done
uganda tuned done
uganda default done
Saved: C:\maternal-mental-health-risk\reports/tables\revision\S2_validation_strategy_table.csv  (32 rows)


,dataset,rf_config,strategy,n_eval,auc,ci_lower,ci_upper,fold_mean_auc,fold_sd_auc,signature_exposure
0,pakistan,tuned,S1 ordinary stratified 5-fold CV,14008,0.568,0.558,0.578,0.568,0.012,0.800
1,pakistan,tuned,S2 group-aware 5-fold CV,14008,0.558,0.548,0.569,0.559,0.017,0.000
2,pakistan,tuned,S3 ordinary random 80/20 split (test AUC),2802,0.568,0.546,0.591,NaN,NaN,0.819
3,pakistan,tuned,S4 group-aware held-out test partition (study ...,2798,0.557,0.536,0.579,NaN,NaN,0.000
4,pakistan,default,S1 ordinary stratified 5-fold CV,14008,0.968,0.965,0.970,0.968,0.003,0.800
5,pakistan,default,S2 group-aware 5-fold CV,14008,0.530,0.519,0.540,0.530,0.012,0.000
6,pakistan,default,S3 ordinary random 80/20 split (test AUC),2802,0.978,0.972,0.982,NaN,NaN,0.819
7,pakistan,default,S4 group-aware held-out test partition (study ...,2798,0.517,0.495,0.537,NaN,NaN,0.000
8,arab5_depression,tuned,S1 ordinary stratified 5-fold CV,1499,0.683,0.656,0.708,0.684,0.034,0.053
9,arab5_depression,tuned,S2 group-aware 5-fold CV,1499,0.683,0.656,0.709,0.684,0.011,0.000


**How to read it / what to write:** the inflation is expected to be largest for the high-capacity (`default`) forest on
datasets with many repeated profiles (Pakistan). If `tuned` shows little inflation, say so explicitly: the 0.56 -> 0.84
contrast in the original draft applies to an unconstrained forest, and the manuscript must state which configuration produced it.